# Classification of complex networks (Lab 7)

## Imports & Support data



In [58]:
! pip install networkx zstandard scikit-learn tabulate requests pandas numpy matplotlib -q

In [59]:
from pathlib import Path
from io import BytesIO
from urllib.parse import urlparse
import re

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import zstandard as zstd
from sklearn import metrics
from tabulate import tabulate

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics.pairwise import euclidean_distances


In [60]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [61]:
datasets = {
    "biological": [
        {
            "name": "celegans_interactomes/wi2007",
            "path": "https://networks.skewed.de/net/celegans_interactomes/files/wi2007.xml.zst"
        },
        {
            "name": "collins_yeast",
            "path": "https://networks.skewed.de/net/collins_yeast/files/collins_yeast.xml.zst"
        },
        {
            "name": "malaria_genes/HVR_1",
            "path": "https://networks.skewed.de/net/malaria_genes/files/HVR_1.xml.zst"
        },
    ],
    "social": [
        {
            "name": "netscience",
            "path": "https://networks.skewed.de/net/netscience/files/netscience.xml.zst"
        },
        {
            "name": "arxiv_authors/HepPh",
            "path": "https://networks.skewed.de/net/arxiv_authors/files/HepPh.xml.zst"
        },
        {
            "name": "flickr_groups",
            "path": "https://networks.skewed.de/net/flickr_groups/files/flickr_groups.xml.zst"
        },
    ],
    "technological": [
        {
            "name": "internet_as",
            "path": "https://networks.skewed.de/net/internet_as/files/internet_as.xml.zst"
        },
        {
            "name": "as_skitter",
            "path": "https://networks.skewed.de/net/as_skitter/files/as_skitter.xml.zst"
        },
        {
            "name": "python_dependency",
            "path": "https://networks.skewed.de/net/python_dependency/files/python_dependency.xml.zst"
        },
    ],
}

## Support Functions

### download_xml_zst

Downloads a `.xml.zst` file from a given URL into a local directory, creating the directory if needed.

It validates that the URL points to an `.xml.zst` file, streams the download in chunks, and returns the saved file path as a `Path` object.


In [5]:
def download_xml_zst(url: str, output_dir: str = "data/raw") -> Path:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    filename = Path(urlparse(url).path).name

    if not filename.endswith(".xml.zst"):
        raise ValueError(f"URL does not point to an .xml.zst file: {filename}")

    file_path = output_path / filename

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    return file_path

### plot_network

Generates a 2D visualization of a graph using a spring layout for node positioning.

It customizes styling (colors, sizes, background), renders nodes and edges with Matplotlib and NetworkX, displays the plot, and returns the computed node positions.


In [6]:
def plot_network(G: nx.Graph):
    pos = nx.spring_layout(
        G,
        seed=42,
        k=0.01,
        iterations=300,
    )

    fig, ax = plt.subplots(figsize=(10, 10))

    background_color = "#cfcfcf"

    fig.patch.set_facecolor(background_color)
    ax.set_facecolor(background_color)

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        edge_color="#5a5a5a",
        width=0.4,
        alpha=0.7,
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=22,
        node_color="#b72b2f",
        edgecolors="#8a1f22",
        linewidths=0.3,
        alpha=0.95,
    )

    ax.set_axis_off()
    plt.tight_layout(pad=0)
    plt.show()

    return pos

### read_xml_zst_to_networkx

Reads a local compressed `.xml.zst` GraphML file and converts it into a NetworkX graph.

It validates the file, decompresses it, patches unsupported GraphML attribute types, optionally removes incompatible tags, and can convert the graph to undirected or relabel nodes as integers.


In [7]:
from pathlib import Path
from io import BytesIO
import re
import zstandard as zstd
import networkx as nx


def read_xml_zst_to_networkx(
    file_path: str | Path,
    relabel_to_int: bool = False,
) -> nx.Graph:
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if file_path.suffixes[-2:] != [".xml", ".zst"]:
        raise ValueError(f"Expected a .xml.zst file, got: {file_path}")

    dctx = zstd.ZstdDecompressor()

    with open(file_path, "rb") as compressed_file:
        with dctx.stream_reader(compressed_file) as reader:
            graphml_data = reader.read()

    graphml_text = graphml_data.decode("utf-8", errors="replace")

    valid_graphml_types = {
        "boolean",
        "int",
        "long",
        "float",
        "double",
        "string",
    }

    def patch_attr_type(match):
        attr_type = match.group(1)
        if attr_type in valid_graphml_types:
            return match.group(0)
        return 'attr.type="string"'

    graphml_text = re.sub(
        r'attr\.type="([^"]+)"',
        patch_attr_type,
        graphml_text,
    )

    graphml_text = re.sub(
        r"<y:.*?</y:.*?>",
        "",
        graphml_text,
        flags=re.DOTALL,
    )

    graph = nx.read_graphml(BytesIO(graphml_text.encode("utf-8")))

    graph = nx.Graph(graph)
    graph.remove_edges_from(nx.selfloop_edges(graph))

    if relabel_to_int:
        graph = nx.convert_node_labels_to_integers(
            graph,
            first_label=0,
            ordering="default",
        )

    return graph

### feature_extractor

Extracts structural metrics from a NetworkX graph and returns them as a dictionary.

It computes node/edge counts, degree moments and variance, clustering, shortest path, assortativity, and the giant component fraction.


In [8]:
def feature_extractor(G:nx.Graph) -> dict[str, float]:
    def moment_of_degree_distribution(G: nx.Graph, m: int) -> float:
        N = len(G)

        if N == 0:
            return 0.0

        total = 0

        for node in G.nodes:
            total += G.degree(node) ** m

        return total / N
    
    def giant_component_fraction(G):
        if G.number_of_nodes() == 0:
            return 0.0
        largest_cc = max(nx.connected_components(G), key=len)
        return len(largest_cc) / G.number_of_nodes()
    
    N = G.number_of_nodes()
    M = G.number_of_edges()

    k1 = moment_of_degree_distribution(G, 1)
    k2 = moment_of_degree_distribution(G, 2)

    variance = k2 - k1**2

    av_cl = nx.average_clustering(G)

    if nx.is_connected(G):
        avg_shortest_path = nx.average_shortest_path_length(G)
    else:
        avg_shortest_path = 0.0

    assortativity = nx.degree_assortativity_coefficient(G)

    return {
        "nodes": N,
        "edges": M,
        "k1": k1,
        "k2": k2,
        "degree_variance": variance,
        "average_clustering": av_cl,
        "average_shortest_path": avg_shortest_path,
        "degree_assortativity": assortativity,
        "giant_component_fraction": giant_component_fraction(G)
    }

### generate_random_networks_from_graph

`generate_random_networks_from_graph` takes the original graph `G` and generates 5 null models, each representing a different hypothesis about the structural mechanisms that could have produced the original network. All models are built with `seed=42` for reproducibility, and the base graph is always converted to undirected and stripped of self-loops before any parameter is extracted.

#### erdos_renyi

Generated with `gnm_random_graph(n, m)`, which places exactly `m` edges uniformly at random across `n` nodes. It is the simplest null model — no degree heterogeneity, no community structure, no preferential attachment. Acts as the baseline: if the original network is not distinguishable from ER, it carries no meaningful structure.

#### configuration_model

Preserves the exact degree sequence of the original graph by wiring stubs randomly. It is the most faithful structural null model since `k1` and the degree distribution are kept intact. Self-loops are removed after generation, which is why edge count varies slightly from the original. If the classifier consistently picks this model, it suggests the degree sequence alone explains most of the original network's topology.

#### barabasi_albert

Grows the network by iteratively adding nodes with `m_ba` edges each, connecting preferentially to high-degree nodes. `m_ba` is derived from `avg_degree / 2` and clamped to valid bounds. This model captures scale-free behavior and hub formation — common in biological and social networks. Edge count will differ from the original since BA is a growth model, not an edge-preserving one.

#### watts_strogatz

Starts from a regular ring lattice with `k_ws` neighbors per node and rewires each edge with probability `0.1`, producing a small-world network with high clustering and short paths. `k_ws` is derived from `avg_degree`, rounded to the nearest even number as required by the model. Useful to test whether the original network exhibits the small-world property.

#### stochastic_block_model

Divides the `n` nodes into 4 equally-sized blocks and assigns within-block probability `p_in = p * 4` and cross-block probability `p_out = p * 0.25`, where `p` is the global edge density of the original. This model captures community structure — if the classifier favors SBM, it suggests the original network has meaningful modular organization that the other models fail to reproduce.

In [69]:
def generate_random_networks_from_graph(G: nx.Graph) -> dict:
    seed = 42

    G_base = G.to_undirected() if G.is_directed() else G.copy()
    G_base.remove_edges_from(nx.selfloop_edges(G_base))

    n = G_base.number_of_nodes()
    m = G_base.number_of_edges()

    avg_degree = (2 * m) / n
    p = (2 * m) / (n * (n - 1))

    degree_sequence = [d for _, d in G_base.degree()]

    ### Create Erdos Renyi Graph
    G_er = nx.gnm_random_graph(n=n, m=m, seed=seed)

    ### Create Configuration Model Graph
    G_conf = nx.configuration_model(degree_sequence, seed=seed)
    G_conf = nx.Graph(G_conf)
    G_conf.remove_edges_from(nx.selfloop_edges(G_conf))

    ### Create Barabasi Albert Model
    m_ba = max(1, min(round(avg_degree / 2), n - 1))
    G_ba = nx.barabasi_albert_graph(n=n, m=m_ba, seed=seed)

    ### Create Watts Strongatz Graph
    k_ws = max(2, round(avg_degree))
    k_ws = k_ws + 1 if k_ws % 2 != 0 else k_ws
    k_ws = min(k_ws, n - 1)
    k_ws = k_ws - 1 if k_ws % 2 != 0 else k_ws
    G_ws = nx.watts_strogatz_graph(n=n, k=k_ws, p=0.1, seed=seed)


    ### Creates Stochastic Block Model Graph
    sizes = [n // 4] * 4
    sizes[-1] += n % 4
    p_in = min(p * 4, 1.0)
    p_out = p * 0.25
    probs = [
        [p_in if i == j else p_out for j in range(4)]
        for i in range(4)
    ]
    G_sbm = nx.stochastic_block_model(sizes=sizes, p=probs, seed=seed)

    return {
        "erdos_renyi": G_er,
        "configuration_model": G_conf,
        "barabasi_albert": G_ba,
        "watts_strogatz": G_ws,
        "stochastic_block_model": G_sbm,
    }

### plot_feature_table

Given a netowrk name and a pandas dataframe, this function will print a well formated table using psql format 

In [68]:
def plot_feature_table(df: pd.DataFrame, network_name: str):
    table = df[df["network"] == network_name].drop(columns="network").reset_index(drop=True)
    print(network_name, end="\n")
    print(
        tabulate(
            df[df["network"]==network_name].drop("network", axis=1), 
            headers='keys',
            tablefmt='psql',
            showindex=False,
            floatfmt=".4f",
            maxcolwidths=[None] * len(table.columns),
        )
    )

### Classifiers

#### naive_bayes_classifier

Applies Gaussian Naive Bayes, which models each feature as an independent Gaussian distribution per class and classifies based on Bayes theorem. Despite the strong independence assumption, it tends to perform surprisingly well in low-sample scenarios. Since we only have 5 training instances (one per generative model), NB is a strong baseline — it doesn't overfit and produces well-calibrated probabilities via `predict_proba`.

#### logistic_regression_classifier

Fits a linear decision boundary in the standardized feature space using logistic regression with L2 regularization. It outputs class probabilities through a softmax and is interpretable — the learned weights directly reflect which features drive the classification. Useful here to understand whether the generative models are linearly separable in metric space.

#### svm_classifier

Uses a Support Vector Machine with an RBF kernel, which maps the feature space into a higher-dimensional space to find non-linear decision boundaries. With few training samples and continuous normalized features, SVM maximizes the margin between classes, making it robust against the noise introduced by the stochastic nature of the generative models (e.g., edges varying across `configuration_model`, `barabasi_albert`). Probabilities are exposed via Platt scaling (`probability=True`).

#### knn_classifier

Classifies the original network by finding its nearest neighbor in the standardized feature space among the 5 generative models. Uses `k=1` since each generative model has exactly one training instance — increasing `k` would mix distinct classes without benefit. Confidence is computed as `1 - (nearest_dist / max_dist)`, normalizing the closest distance against the furthest candidate, giving an interpretable proximity score in `[0, 1]`. Also returns the raw distance for additional analysis.

In [65]:
def _run_classifier(table, clf):
    """Base runner reutilizável para qualquer classificador sklearn."""
    original = table[table["type"] == "original"]
    random   = table[table["type"] != "original"]

    feature_cols = [col for col in table.columns if col != "type"]
    y_train      = random["type"].values

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(random[feature_cols].values)
    X_test_scaled  = scaler.transform(original[feature_cols].values)

    clf.fit(X_train_scaled, y_train)
    predicted = clf.predict(X_test_scaled)[0]
    proba     = clf.predict_proba(X_test_scaled)[0] if hasattr(clf, "predict_proba") else None

    confidence = round(float(max(proba)), 4) if proba is not None else None
    return predicted, confidence

def naive_bayes_classifier(df: pd.DataFrame, network_name: str):
    table = df[df["network"] == network_name].drop(columns="network").reset_index(drop=True)
    predicted, confidence = _run_classifier(table, GaussianNB())
    return {
        "network":    network_name,
        "predicted":  predicted,
        "confidence": confidence,  # P(classe | features)
    }


def logistic_regression_classifier(df: pd.DataFrame, network_name: str):
    table = df[df["network"] == network_name].drop(columns="network").reset_index(drop=True)
    clf   = LogisticRegression(max_iter=1000, random_state=42)
    predicted, confidence = _run_classifier(table, clf)
    return {
        "network":    network_name,
        "predicted":  predicted,
        "confidence": confidence,
    }


def svm_classifier(df: pd.DataFrame, network_name: str):
    table = df[df["network"] == network_name].drop(columns="network").reset_index(drop=True)
    # probability=True para expor predict_proba via Platt scaling
    clf   = SVC(kernel="rbf", probability=True, random_state=42)
    predicted, confidence = _run_classifier(table, clf)
    return {
        "network":    network_name,
        "predicted":  predicted,
        "confidence": confidence,
    }

def knn_classifier(df: pd.DataFrame, network_name: str):
    table = df[df["network"] == network_name].drop(columns="network").reset_index(drop=True)

    original = table[table["type"] == "original"]
    random   = table[table["type"] != "original"]

    feature_cols = [col for col in table.columns if col != "type"]
    y_train      = random["type"].values

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(random[feature_cols].values)
    X_test_scaled  = scaler.transform(original[feature_cols].values)

    knn = KNeighborsClassifier(n_neighbors=1, metric="euclidean")
    knn.fit(X_train_scaled, y_train)

    predicted    = knn.predict(X_test_scaled)[0]
    dists, _     = knn.kneighbors(X_test_scaled, return_distance=True)
    nearest_dist = dists[0][0]

    # Distâncias do teste para TODOS os pontos de treino
    all_dists  = euclidean_distances(X_test_scaled, X_train_scaled)[0]
    max_dist   = all_dists.max()

    confidence = round(1 - (nearest_dist / (max_dist + 1e-9)), 4)

    return {
        "network":    network_name,
        "predicted":  predicted,
        "distance":   round(nearest_dist, 4),
        "confidence": confidence,
    }

## Biological Networks



In [12]:
for n in datasets["biological"]:
    temp = download_xml_zst(n["path"])
    n["networks"]  = {
        "original": read_xml_zst_to_networkx(
            temp,
            relabel_to_int=True
        )
    }
    print(
        n["networks"]["original"]
    )

Graph named 'celegans_interactomes (wi2007)' with 1496 nodes and 1714 edges
Graph named 'collins_yeast' with 1622 nodes and 9070 edges
Graph named 'malaria_genes (HVR_1)' with 307 nodes and 2812 edges


In [13]:
for n in datasets["biological"]:
    G_temp = n["networks"]["original"]

    n["networks"].update(
        generate_random_networks_from_graph(G_temp)
    )

In [14]:
features = []
for n in datasets["biological"]:
    for key_ng in n["networks"]:
        tmp = feature_extractor(n["networks"][key_ng])
        tmp.update(
            {
                "type": key_ng,
                "network": n["name"]
            }
        )
        features.append(tmp)

In [15]:
df =  pd.DataFrame(features)
df = df[list(df.columns[-2:]) + list(df.columns[:-2])]

In [ ]:
plot_feature_table(df, datasets["biological"][0]["name"])

celegans_interactomes/wi2007
+------------------------+---------+---------+--------+---------+-------------------+----------------------+-------------------------+------------------------+----------------------------+
| type                   |   nodes |   edges |     k1 |      k2 |   degree_variance |   average_clustering |   average_shortest_path |   degree_assortativity |   giant_component_fraction |
|------------------------+---------+---------+--------+---------+-------------------+----------------------+-------------------------+------------------------+----------------------------|
| original               |    1496 |    1714 | 2.2914 | 24.4158 |           19.1651 |               0.0131 |                  0.0000 |                -0.1984 |                     0.7406 |
| erdos_renyi            |    1496 |    1714 | 2.2914 |  7.4318 |            2.1811 |               0.0025 |                  0.0000 |                 0.0022 |                     0.8690 |
| configuration_model    |

In [17]:
plot_feature_table(df, datasets["biological"][1]["name"])

collins_yeast
+------------------------+---------+---------+---------+----------+-------------------+----------------------+-------------------------+------------------------+----------------------------+
| type                   |   nodes |   edges |      k1 |       k2 |   degree_variance |   average_clustering |   average_shortest_path |   degree_assortativity |   giant_component_fraction |
|------------------------+---------+---------+---------+----------+-------------------+----------------------+-------------------------+------------------------+----------------------------|
| original               |    1622 |    9070 | 11.1837 | 388.4168 |          263.3411 |               0.5546 |                  0.0000 |                 0.6057 |                     0.6190 |
| erdos_renyi            |    1622 |    9070 | 11.1837 | 135.9334 |           10.8577 |               0.0065 |                  3.3346 |                 0.0157 |                     1.0000 |
| configuration_model    |    1

In [18]:
plot_feature_table(df, datasets["biological"][2]["name"])

malaria_genes/HVR_1
+------------------------+---------+---------+---------+----------+-------------------+----------------------+-------------------------+------------------------+----------------------------+
| type                   |   nodes |   edges |      k1 |       k2 |   degree_variance |   average_clustering |   average_shortest_path |   degree_assortativity |   giant_component_fraction |
|------------------------+---------+---------+---------+----------+-------------------+----------------------+-------------------------+------------------------+----------------------------|
| original               |     307 |    2812 | 18.3192 | 457.7785 |          122.1847 |               0.5702 |                  3.5191 |                 0.5991 |                     1.0000 |
| erdos_renyi            |     307 |    2812 | 18.3192 | 351.8046 |           16.2108 |               0.0582 |                  2.2557 |                -0.0299 |                     1.0000 |
| configuration_model    

In [62]:
print(knn_classifier(df, datasets["biological"][0]["name"]))
print(naive_bayes_classifier(df, datasets["biological"][0]["name"]))
print(logistic_regression_classifier(df, datasets["biological"][0]["name"]))
print(svm_classifier(df, datasets["biological"][0]["name"]))

{'network': 'celegans_interactomes/wi2007', 'predicted': 'configuration_model', 'distance': np.float64(5.1325), 'confidence': np.float64(0.3603)}
{'network': 'celegans_interactomes/wi2007', 'predicted': np.str_('configuration_model'), 'confidence': 1.0}
{'network': 'celegans_interactomes/wi2007', 'predicted': 'configuration_model', 'confidence': 0.9334}
{'network': 'celegans_interactomes/wi2007', 'predicted': 'configuration_model', 'confidence': 0.2015}


In [63]:
print(knn_classifier(df, datasets["biological"][1]["name"]))
print(naive_bayes_classifier(df, datasets["biological"][1]["name"]))
print(logistic_regression_classifier(df, datasets["biological"][1]["name"]))
print(svm_classifier(df, datasets["biological"][1]["name"]))

{'network': 'collins_yeast', 'predicted': 'configuration_model', 'distance': np.float64(256.3413), 'confidence': np.float64(0.0101)}
{'network': 'collins_yeast', 'predicted': np.str_('configuration_model'), 'confidence': 1.0}
{'network': 'collins_yeast', 'predicted': 'configuration_model', 'confidence': 1.0}
{'network': 'collins_yeast', 'predicted': 'watts_strogatz', 'confidence': 0.2}


In [64]:
print(knn_classifier(df, datasets["biological"][2]["name"]))
print(naive_bayes_classifier(df, datasets["biological"][2]["name"]))
print(logistic_regression_classifier(df, datasets["biological"][2]["name"]))
print(svm_classifier(df, datasets["biological"][2]["name"]))

{'network': 'malaria_genes/HVR_1', 'predicted': 'watts_strogatz', 'distance': np.float64(37.1915), 'confidence': np.float64(0.0791)}
{'network': 'malaria_genes/HVR_1', 'predicted': np.str_('watts_strogatz'), 'confidence': 1.0}
{'network': 'malaria_genes/HVR_1', 'predicted': 'barabasi_albert', 'confidence': 0.9134}
{'network': 'malaria_genes/HVR_1', 'predicted': 'watts_strogatz', 'confidence': 0.2}
